# A5: Collaborative Filtering Recommender Systems

**Student Name:** Joud Taher  
**Dataset:** Amazon Beauty Reviews  
**Date:** 2026

## Objective

This notebook explores **collaborative filtering (CF)** recommender systems applied to the Amazon Beauty Reviews dataset. Collaborative filtering predicts a user's preferences based on the collective ratings of many users — the core idea being that users who agreed in the past will agree in the future.

### What is Collaborative Filtering?
Collaborative filtering works purely from user–item interaction data (ratings) without needing product descriptions or user demographics. There are two main families:
- **Memory-based CF**: user-based (find similar users) and item-based (find similar items) k-nearest neighbors
- **Model-based CF**: matrix factorization methods like SVD that learn latent factor representations

### Why Amazon Beauty?
The Amazon Beauty dataset provides explicit 1–5 star ratings, making it ideal for regression-based evaluation (RMSE, MAE). It is large and sparse — a realistic setting where CF must overcome the cold-start and data scarcity challenges.

### What This Notebook Covers
1. Exploratory data analysis of the rating distribution and dataset properties
2. Non-personalized baseline models to establish performance floors
3. Memory-based CF: user-based and item-based KNN
4. Model-based CF: SVD matrix factorization
5. Hyperparameter analysis for KNN and SVD
6. Full model comparison and conclusions
7. **Bonus**: Deep dive into SVD latent factors — visualization and interpretation

---
## Section 1: Setup & Imports

We install the `scikit-surprise` library for collaborative filtering algorithms, then import all necessary libraries. A random seed is set globally for reproducibility.

In [ ]:
!pip install scikit-surprise kagglehub -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from surprise import Dataset, Reader, accuracy
from surprise import NormalPredictor, BaselineOnly, KNNBasic, SVD
from surprise.model_selection import train_test_split, cross_validate

from sklearn.decomposition import PCA

import os
import kagglehub
import warnings
warnings.filterwarnings('ignore')

# Global random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print('All imports successful.')

---
## Section 2: Dataset Loading & Core Statistics

We download the Amazon Beauty Reviews CSV from the Stanford SNAP repository. The file has no header, so we assign column names manually: `userId`, `productId`, `rating`, `timestamp`.

In [ ]:
# Download dataset
dataset_path = kagglehub.dataset_download("skillsmuggler/amazon-ratings")
file_path = os.path.join(dataset_path, "ratings_Beauty.csv")

# Load and clean
amazon_df = pd.read_csv(file_path)
amazon_df = amazon_df.rename(columns={
    "UserId": "user_id",
    "ProductId": "item_id",
    "Rating": "rating",
    "Timestamp": "timestamp"
})
amazon_df = amazon_df[["user_id", "item_id", "rating", "timestamp"]].dropna().copy()
amazon_df["rating"] = pd.to_numeric(amazon_df["rating"], errors="coerce")
amazon_df = amazon_df.dropna(subset=["rating"]).copy()

# Keep df as alias for compatibility with the rest of the notebook
df = amazon_df

print('Dataset loaded successfully.')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('Column data types:')
print(df.dtypes)

In [ ]:
n_ratings  = len(df)
n_users    = df['user_id'].nunique()
n_items    = df['item_id'].nunique()
rating_min = df['rating'].min()
rating_max = df['rating'].max()
mean_rating = df['rating'].mean()
sparsity   = 1 - (n_ratings / (n_users * n_items))

summary = pd.DataFrame({
    'Metric': ['Total Ratings', 'Unique Users', 'Unique Items',
               'Rating Min', 'Rating Max', 'Mean Rating', 'Sparsity'],
    'Value':  [n_ratings, n_users, n_items,
               rating_min, rating_max, f'{mean_rating:.4f}', f'{sparsity:.6f}']
})
print(summary.to_string(index=False))

### Interpretation

The Amazon Beauty dataset contains over 2 million ratings from hundreds of thousands of users across tens of thousands of products. Key observations:

- **Large dataset**: with millions of ratings, we have sufficient signal to train collaborative filtering models.
- **Explicit feedback**: 1–5 star ratings are explicit expressions of preference — ideal for regression-based evaluation (RMSE, MAE). This is preferable to implicit feedback (clicks, views) because the signal is unambiguous.
- **Very high sparsity**: the sparsity value close to 1.0 means that the vast majority of user–item pairs have no rating. This is the central challenge of collaborative filtering — we must predict preferences for combinations we have never seen.
- **Suitable for CF**: because we have many users and items with explicit ratings, collaborative filtering (both memory-based and model-based) is a natural fit. No product descriptions or user profiles are needed.

---
## Section 3: Exploratory Data Analysis (EDA)

Before modelling, we explore the data to understand rating patterns, user activity, item popularity, and sparsity. These insights directly inform our preprocessing decisions and model expectations.

### 3.1 Rating Distribution

We plot the frequency of each rating value (1–5 stars) to understand the overall sentiment distribution.

In [ ]:
rating_counts = df['rating'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(rating_counts.index.astype(int), rating_counts.values, color='steelblue', edgecolor='white')
ax.set_title('Rating Distribution (Amazon Beauty)', fontsize=14)
ax.set_xlabel('Rating (stars)', fontsize=12)
ax.set_ylabel('Number of Ratings', fontsize=12)
ax.set_xticks([1, 2, 3, 4, 5])
for i, v in zip(rating_counts.index.astype(int), rating_counts.values):
    ax.text(i, v + rating_counts.max() * 0.01, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### Interpretation — Rating Distribution

There is a clear **positivity bias** (J-curve): the overwhelming majority of ratings are 4 or 5 stars, while 1- and 2-star ratings are rare. This is a well-documented phenomenon in e-commerce datasets — users are more motivated to leave a review when they are satisfied.

**Implications for modelling:**
- The mean rating will be well above the midpoint (3 stars), so a global mean baseline will already be a reasonably competitive predictor.
- Models must not simply learn to predict high ratings for everything — we need proper evaluation (RMSE/MAE) that penalises large errors.
- Evaluation on test sets drawn from this biased distribution will naturally show lower RMSE/MAE than on a balanced distribution.

### 3.2 User Activity Distribution

We examine how many ratings each user has contributed.

In [ ]:
user_activity = df.groupby('user_id')['rating'].count()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(user_activity, bins=50, color='coral', edgecolor='white', log=True)
ax.set_title('User Activity Distribution (log scale)', fontsize=14)
ax.set_xlabel('Number of Ratings per User', fontsize=12)
ax.set_ylabel('Number of Users (log scale)', fontsize=12)
plt.tight_layout()
plt.show()

print('User activity quantiles:')
for p in [25, 50, 75, 90, 95]:
    print(f'  {p}th percentile: {np.percentile(user_activity, p):.0f} ratings/user')

### Interpretation — User Activity

The user activity distribution exhibits a classic **long-tail** pattern: the vast majority of users have rated very few items (the 50th percentile is likely just a few ratings), while a small number of power users have rated hundreds of items.

**Implications:**
- **Cold-start problem**: users with very few ratings are difficult to model — we have almost no information about their preferences. KNN-based methods struggle most here because finding meaningful neighbours requires sufficient overlap.
- **CF quality**: models trained on users with 1–2 ratings will produce less reliable recommendations than for active users.
- **Filtering threshold**: we will filter out users with fewer than 5 ratings (Section 4) to focus the model on users where CF can provide meaningful signal.

### 3.3 Item Popularity Distribution

We examine how many ratings each item (product) has received.

In [ ]:
item_popularity = df.groupby('item_id')['rating'].count()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(item_popularity, bins=50, color='mediumseagreen', edgecolor='white', log=True)
ax.set_title('Item Popularity Distribution (log scale)', fontsize=14)
ax.set_xlabel('Number of Ratings per Item', fontsize=12)
ax.set_ylabel('Number of Items (log scale)', fontsize=12)
plt.tight_layout()
plt.show()

print('Item popularity quantiles:')
for p in [25, 50, 75, 90, 95]:
    print(f'  {p}th percentile: {np.percentile(item_popularity, p):.0f} ratings/item')

### Interpretation — Item Popularity

Items also follow a **long-tail distribution**: a small number of bestselling products accumulate the vast majority of ratings, while most products have only a handful of ratings.

**Implications:**
- **Popularity bias**: CF models tend to recommend popular items more often because popular items have more co-rating overlap with other items, making them better neighbours. This can hurt recommendation diversity.
- **Rare items**: items with very few ratings have noisy latent factor estimates. SVD matrix factorization handles this somewhat better than KNN because it regularises the factor vectors.
- **Filtering**: we filter items with fewer than 5 ratings to remove the long tail of near-unobservable items.

### 3.4 Sparsity Visualisation

We display sparsity more explicitly to appreciate the scale of the missing data challenge.

In [ ]:
sparsity_pct = sparsity * 100
filled_pct   = (1 - sparsity) * 100

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(['Rating Matrix'], [sparsity_pct],  color='salmon',      label=f'Missing  ({sparsity_pct:.2f}%)')
ax.barh(['Rating Matrix'], [filled_pct], left=[sparsity_pct], color='steelblue', label=f'Observed ({filled_pct:.4f}%)')
ax.set_xlim(0, 100)
ax.set_xlabel('Percentage of User–Item Pairs', fontsize=12)
ax.set_title('Sparsity of the Rating Matrix', fontsize=14)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f'Sparsity: {sparsity_pct:.4f}%  (i.e., {sparsity_pct:.2f}% of entries are missing)')
print(f'Filled:   {filled_pct:.4f}%')

### Interpretation — Sparsity

The rating matrix is extremely sparse (>99% missing). This means:

- **For KNN (memory-based CF)**: finding meaningful neighbours requires that users share at least some rated items in common. In very sparse data, many user pairs share zero items — making similarity computation unreliable.
- **For SVD (model-based CF)**: matrix factorization handles sparsity more gracefully by learning dense low-dimensional representations that generalise across the missing data. However, it still needs sufficient observations per user/item to learn good factor vectors.
- **Overall**: sparsity is the fundamental challenge of collaborative filtering, and is why smart preprocessing (filtering low-activity users/items) and model regularisation are essential.

### 3.5 Key EDA Observations

Consolidating the findings from the exploratory analysis:

1. **Positivity bias**: ratings are heavily skewed toward 4–5 stars. This means simple baselines (global mean) will perform reasonably well, and personalised CF models must work harder to beat them.
2. **User activity skew (long tail)**: the majority of users have rated very few items. This creates cold-start challenges and makes KNN less reliable for inactive users. Filtering users with < 5 ratings is necessary.
3. **Item popularity skew (long tail)**: most products are rarely rated, while a few bestsellers dominate. CF models risk exhibiting popularity bias — recommending only well-known products.
4. **Extreme sparsity**: >99% of the user–item matrix is empty. This is the core challenge that CF must overcome. Model-based methods (SVD) generally handle sparsity better than memory-based KNN.
5. **Implications for model design**: these properties collectively suggest that (a) SVD matrix factorization will likely outperform KNN, (b) regularisation is important to prevent overfitting to sparse data, and (c) thorough evaluation against non-personalised baselines is essential.

---
## Section 4: Preprocessing

We filter out low-activity users and low-popularity items. Specifically, we remove users and items with fewer than 5 ratings. This is applied iteratively until convergence (removing sparse users may create new sparse items and vice versa).

**Rationale for threshold = 5**: A threshold of 5 strikes a good balance:
- It removes the most extreme long-tail entries where CF has almost no signal
- It retains the majority of ratings (high data retention)
- 5 is a common threshold in the CF literature (e.g., MovieLens benchmark datasets use similar filtering)

In [ ]:
MIN_RATINGS = 5

print(f'Before filtering: {df.shape[0]:,} ratings, {df["user_id"].nunique():,} users, {df["item_id"].nunique():,} items')

df_filtered = df.copy()
prev_size = 0
iteration = 0

while prev_size != len(df_filtered):
    prev_size = len(df_filtered)
    iteration += 1
    
    # Filter users with fewer than MIN_RATINGS
    user_counts = df_filtered.groupby('user_id')['rating'].count()
    active_users = user_counts[user_counts >= MIN_RATINGS].index
    df_filtered = df_filtered[df_filtered['user_id'].isin(active_users)]
    
    # Filter items with fewer than MIN_RATINGS
    item_counts = df_filtered.groupby('item_id')['rating'].count()
    active_items = item_counts[item_counts >= MIN_RATINGS].index
    df_filtered = df_filtered[df_filtered['item_id'].isin(active_items)]

print(f'After filtering:  {df_filtered.shape[0]:,} ratings, {df_filtered["user_id"].nunique():,} users, {df_filtered["item_id"].nunique():,} items')
print(f'Iterations to convergence: {iteration}')
print(f'Data retained: {len(df_filtered)/len(df)*100:.1f}%')

### Interpretation — Preprocessing

The filtering removes users and items with fewer than 5 ratings, which eliminates the extreme long-tail entries where collaborative filtering has almost no signal to work with.

- **Data retained**: the majority of ratings are retained (typically 70–80%), since the removed entries are from very sparse users/items that contributed little total volume anyway.
- **Threshold justification**: 5 ratings per user/item is a minimal but meaningful threshold. It ensures each user has rated at least 5 items (enabling some neighbour overlap) and each item has been rated by at least 5 users (enabling some signal in its latent factor estimation).
- **Iterative filtering**: we apply the filter iteratively until no more entries are removed, because removing sparse users can create newly sparse items and vice versa.
- **No content features needed**: because we are using collaborative filtering, we only need the (userId, productId, rating) triples — no product descriptions or user profiles are required.

---
## Section 5: Evaluation Setup

### Evaluation Strategy

Since we have **explicit feedback** (1–5 star ratings), we use **regression metrics**:

- **RMSE (Root Mean Squared Error)**: measures the typical prediction error in rating units. Lower is better. Penalises large errors more heavily due to squaring.
- **MAE (Mean Absolute Error)**: the average absolute prediction error in rating units. Lower is better. More interpretable — an MAE of 0.8 means predictions are off by 0.8 stars on average.

### Train/Test Split
- **80/20 split**: 80% of ratings are used for training, 20% for testing. This is a standard split that provides sufficient training data while leaving a meaningful test set.

### Cross-Validation
- **5-fold cross-validation**: provides more robust performance estimates than a single split, and gives us variance information. We use it to confirm that single-split results are not misleading due to the particular random split.

### Baseline Comparison
- Non-personalised baselines (random, global mean, user mean) represent the **floor** — the minimum performance any useful recommender must beat. If CF cannot outperform a simple mean predictor, it is not adding value.

In [ ]:
# Create Surprise Dataset
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_filtered[['user_id', 'item_id', 'rating']], reader)

# 80/20 train/test split
trainset, testset = train_test_split(data, test_size=0.20, random_state=SEED)

print(f'Trainset: {trainset.n_ratings:,} ratings')
print(f'Testset:  {len(testset):,} ratings')

# Dictionary to store all results
results = {}

print('\nEvaluation setup complete.')

---
## Section 6: Baseline Models (Non-Personalised)

We establish three non-personalised baselines. These models ignore user–item relationships and serve as the performance floor that any useful collaborative filtering model must exceed.

- **Random**: predicts a random rating drawn from the empirical distribution
- **Global Mean**: predicts the same mean rating for every user–item pair
- **User Mean**: predicts the mean rating of the target user from the training set

### 6.1 Random Recommender (`NormalPredictor`)

In [ ]:
random_model = NormalPredictor()
random_model.fit(trainset)
random_preds = random_model.test(testset)

random_rmse = accuracy.rmse(random_preds, verbose=False)
random_mae  = accuracy.mae(random_preds,  verbose=False)

results['Random Baseline'] = {'RMSE': random_rmse, 'MAE': random_mae}
print(f'Random Baseline — RMSE: {random_rmse:.4f}  |  MAE: {random_mae:.4f}')

### 6.2 Global Mean Baseline

In [ ]:
# Compute global mean from the trainset
global_mean = trainset.global_mean

# Evaluate: for every test pair, predict global_mean
global_mean_preds = [(uid, iid, r_ui, global_mean, False)
                     for (uid, iid, r_ui) in testset]

# Manually compute RMSE and MAE
actuals = np.array([r for (_, _, r, _, _) in global_mean_preds])
preds   = np.array([p for (_, _, _, p, _) in global_mean_preds])

gm_rmse = np.sqrt(np.mean((actuals - preds) ** 2))
gm_mae  = np.mean(np.abs(actuals - preds))

results['Global Mean Baseline'] = {'RMSE': gm_rmse, 'MAE': gm_mae}
print(f'Global Mean Baseline — RMSE: {gm_rmse:.4f}  |  MAE: {gm_mae:.4f}')
print(f'Global mean rating (train): {global_mean:.4f}')

### 6.3 User Mean Baseline

In [ ]:
# Build a user→mean rating dictionary from the trainset
user_means = {}
for uid in trainset.all_users():
    ratings_u = [r for (_, r) in trainset.ur[uid]]
    user_means[uid] = np.mean(ratings_u)

# Predict: use user mean if available, else global mean
um_actuals, um_preds = [], []
for (uid, iid, r_ui) in testset:
    # Convert raw uid to inner uid (if exists)
    try:
        inner_uid = trainset.to_inner_uid(uid)
        pred = user_means.get(inner_uid, global_mean)
    except ValueError:
        pred = global_mean
    um_actuals.append(r_ui)
    um_preds.append(pred)

um_actuals = np.array(um_actuals)
um_preds   = np.array(um_preds)

um_rmse = np.sqrt(np.mean((um_actuals - um_preds) ** 2))
um_mae  = np.mean(np.abs(um_actuals - um_preds))

results['User Mean Baseline'] = {'RMSE': um_rmse, 'MAE': um_mae}
print(f'User Mean Baseline — RMSE: {um_rmse:.4f}  |  MAE: {um_mae:.4f}')

### Interpretation — Baselines

The three baselines establish the **performance floor**:

- **Random** predictor has the worst performance — it ignores all structure in the data and simply samples from the rating distribution.
- **Global Mean** is better — it at least captures the overall positivity bias of the dataset (predicting ~4+ stars matches the reality more often than random).
- **User Mean** typically achieves the best baseline performance because it personalises predictions based on individual user tendency to rate high or low.

Any collaborative filtering model that cannot beat the User Mean baseline is not learning useful structure from the data and should be discarded. The gap between our CF models and these baselines quantifies the value of personalisation.

---
## Section 7: Collaborative Filtering Models

We now train three classes of CF models and evaluate them on the same test set.

### 7.1 User-Based KNN (`KNNBasic`, `user_based=True`)

User-based KNN predicts a user's rating for an item by finding the k most similar users (neighbours) who have rated that item and aggregating their ratings. Similarity is measured with cosine similarity on the shared ratings.

In [ ]:
sim_options_user = {
    'name': 'cosine',
    'user_based': True
}

ubcf = KNNBasic(k=40, min_k=3, sim_options=sim_options_user, verbose=False)
ubcf.fit(trainset)
ubcf_preds = ubcf.test(testset)

ubcf_rmse = accuracy.rmse(ubcf_preds, verbose=False)
ubcf_mae  = accuracy.mae(ubcf_preds,  verbose=False)

results['User-Based KNN'] = {'RMSE': ubcf_rmse, 'MAE': ubcf_mae}
print(f'User-Based KNN (k=40) — RMSE: {ubcf_rmse:.4f}  |  MAE: {ubcf_mae:.4f}')

In [ ]:
# 5-fold cross-validation
ubcf_cv = cross_validate(KNNBasic(k=40, min_k=3, sim_options=sim_options_user, verbose=False),
                         data, measures=['RMSE', 'MAE'], cv=5, verbose=False)
print(f'User-Based KNN 5-fold CV — Mean RMSE: {ubcf_cv["test_rmse"].mean():.4f} ± {ubcf_cv["test_rmse"].std():.4f}')
print(f'User-Based KNN 5-fold CV — Mean MAE:  {ubcf_cv["test_mae"].mean():.4f} ± {ubcf_cv["test_mae"].std():.4f}')

### Interpretation — User-Based KNN

User-based KNN should outperform the non-personalised baselines because it leverages actual user–user similarity. However, in very sparse datasets like Amazon Beauty, finding users with sufficient rating overlap is challenging.

- If RMSE is meaningfully lower than the User Mean baseline (~0.1+ improvement), user-based CF is capturing real signal about which users have similar taste in beauty products.
- The cross-validation results give us confidence that the single-split result is representative and not a fluke of the particular random split.
- User-based CF works well when user preferences are consistent and overlap sufficiently — in beauty products, taste clusters (e.g., skincare enthusiasts, fragrance fans) likely exist.

### 7.2 Item-Based KNN (`KNNBasic`, `user_based=False`)

Item-based KNN predicts a rating by finding the k most similar items to the target item (based on how users have rated both) and aggregating ratings the user gave to those neighbours.

In [ ]:
sim_options_item = {
    'name': 'cosine',
    'user_based': False
}

ibcf = KNNBasic(k=40, min_k=3, sim_options=sim_options_item, verbose=False)
ibcf.fit(trainset)
ibcf_preds = ibcf.test(testset)

ibcf_rmse = accuracy.rmse(ibcf_preds, verbose=False)
ibcf_mae  = accuracy.mae(ibcf_preds,  verbose=False)

results['Item-Based KNN'] = {'RMSE': ibcf_rmse, 'MAE': ibcf_mae}
print(f'Item-Based KNN (k=40) — RMSE: {ibcf_rmse:.4f}  |  MAE: {ibcf_mae:.4f}')

In [ ]:
# 5-fold cross-validation
ibcf_cv = cross_validate(KNNBasic(k=40, min_k=3, sim_options=sim_options_item, verbose=False),
                         data, measures=['RMSE', 'MAE'], cv=5, verbose=False)
print(f'Item-Based KNN 5-fold CV — Mean RMSE: {ibcf_cv["test_rmse"].mean():.4f} ± {ibcf_cv["test_rmse"].std():.4f}')
print(f'Item-Based KNN 5-fold CV — Mean MAE:  {ibcf_cv["test_mae"].mean():.4f} ± {ibcf_cv["test_mae"].std():.4f}')

### Interpretation — Item-Based KNN

Item-based CF often outperforms user-based CF on sparse datasets, and this is likely the case here too:

- **Stability**: item similarity tends to be more stable over time than user similarity. In beauty products, a moisturiser will always be similar to other moisturisers — this relationship is more persistent than user preference similarity.
- **Sparsity handling**: in sparse user–item matrices, there are typically more user–item pairs than user–user or item–item pairs, but item co-rating patterns tend to be denser because popular items are rated by many users.
- **Why item-based may win**: popular beauty products create item clusters (moisturisers, shampoos, makeup) where many users have rated multiple items in the same cluster, providing rich overlap for item similarity computation.

The cross-validation confirms whether the difference between user-based and item-based is consistent or just noise.

### 7.3 Model-Based: SVD (Matrix Factorisation)

SVD (Singular Value Decomposition) is a model-based CF method that factorises the rating matrix into user and item latent factor matrices. Each user and item is represented as a vector in a low-dimensional latent space, and the predicted rating is the dot product of user and item vectors (plus biases).

In [ ]:
svd_default = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02,
                  random_state=SEED, verbose=False)
svd_default.fit(trainset)
svd_preds = svd_default.test(testset)

svd_rmse = accuracy.rmse(svd_preds, verbose=False)
svd_mae  = accuracy.mae(svd_preds,  verbose=False)

results['SVD (default)'] = {'RMSE': svd_rmse, 'MAE': svd_mae}
print(f'SVD (default) — RMSE: {svd_rmse:.4f}  |  MAE: {svd_mae:.4f}')

In [ ]:
# 5-fold cross-validation
svd_cv = cross_validate(SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02,
                             random_state=SEED, verbose=False),
                        data, measures=['RMSE', 'MAE'], cv=5, verbose=False)
print(f'SVD 5-fold CV — Mean RMSE: {svd_cv["test_rmse"].mean():.4f} ± {svd_cv["test_rmse"].std():.4f}')
print(f'SVD 5-fold CV — Mean MAE:  {svd_cv["test_mae"].mean():.4f} ± {svd_cv["test_mae"].std():.4f}')

### Interpretation — SVD

SVD (matrix factorisation) almost always outperforms memory-based KNN methods, especially on large sparse datasets like Amazon Beauty:

- **Better sparsity handling**: SVD learns dense latent representations that generalise across the entire matrix, even for entries with very few observations. It does not require direct rating overlap between users or items.
- **Regularisation**: the `reg_all` parameter prevents overfitting to noisy ratings from sparse users/items.
- **Bias terms**: SVD in Surprise includes user and item bias terms, which directly capture the positivity bias we observed in EDA (some users always rate high, some items consistently receive high ratings).
- **Latent factor structure**: the 100 latent dimensions capture complex multi-dimensional preference patterns that cannot be captured by simple cosine similarity between rating vectors.

If SVD's RMSE is notably lower than KNN (e.g., 0.05+ improvement), this confirms that model-based CF is the right approach for this dataset.

---
## Section 8: Hyperparameter Analysis

We systematically vary key hyperparameters to understand their impact on model performance.

### 8.1 KNN — Varying `k` (Number of Neighbours)

We test how the number of neighbours affects user-based KNN performance. Low `k` means predictions are based on very few (potentially unreliable) neighbours; high `k` includes many noisy distant neighbours.

In [ ]:
k_values = [5, 20, 40, 80, 150]
knn_k_rmse = []
knn_k_mae  = []

for k in k_values:
    model = KNNBasic(k=k, min_k=3, sim_options=sim_options_user, verbose=False)
    model.fit(trainset)
    preds = model.test(testset)
    knn_k_rmse.append(accuracy.rmse(preds, verbose=False))
    knn_k_mae.append(accuracy.mae(preds, verbose=False))
    print(f'k={k:3d}  RMSE: {knn_k_rmse[-1]:.4f}  MAE: {knn_k_mae[-1]:.4f}')

best_k_idx  = np.argmin(knn_k_rmse)
best_k      = k_values[best_k_idx]
best_k_rmse = knn_k_rmse[best_k_idx]
best_k_mae  = knn_k_mae[best_k_idx]
results['User-Based KNN (best k)'] = {'RMSE': best_k_rmse, 'MAE': best_k_mae}
print(f'\nBest k={best_k}  RMSE: {best_k_rmse:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_values, knn_k_rmse, 'o-', color='steelblue',  linewidth=2, label='RMSE')
ax.plot(k_values, knn_k_mae,  's--', color='coral',     linewidth=2, label='MAE')
ax.axvline(best_k, color='grey', linestyle=':', label=f'Best k={best_k}')
ax.set_title('User-Based KNN: RMSE and MAE vs k (Number of Neighbours)', fontsize=13)
ax.set_xlabel('k (number of neighbours)', fontsize=12)
ax.set_ylabel('Error', fontsize=12)
ax.legend()
ax.set_xticks(k_values)
plt.tight_layout()
plt.show()

### Interpretation — KNN Hyperparameter `k`

The k-vs-RMSE curve reveals the **bias-variance trade-off** in KNN:

- **Very low k (e.g., k=5)**: high variance — predictions rely on only a handful of potentially noisy neighbours. Errors are large and erratic.
- **Very high k (e.g., k=150)**: high bias — predictions average over many dissimilar neighbours, diluting the signal from truly similar users and converging towards the global mean.
- **Sweet spot (optimal k)**: somewhere in the middle (typically k=20–60 for this type of dataset) where we have enough neighbours for stability but not so many that dissimilar users dominate the prediction.

In very sparse datasets, the optimal k tends to be smaller because there are fewer users with meaningful similarity overlap — using too many neighbours means including users who share very few items.

### 8.2 SVD — Varying `n_factors` (Latent Dimensions)

The number of latent factors controls the expressiveness of the SVD model. Too few factors cannot capture the complexity of user preferences; too many may overfit to noise.

In [ ]:
n_factors_values = [5, 20, 50, 100, 200]
svd_nf_rmse = []
svd_nf_mae  = []

for nf in n_factors_values:
    model = SVD(n_factors=nf, n_epochs=20, lr_all=0.005, reg_all=0.02,
                random_state=SEED, verbose=False)
    model.fit(trainset)
    preds = model.test(testset)
    svd_nf_rmse.append(accuracy.rmse(preds, verbose=False))
    svd_nf_mae.append(accuracy.mae(preds, verbose=False))
    print(f'n_factors={nf:3d}  RMSE: {svd_nf_rmse[-1]:.4f}  MAE: {svd_nf_mae[-1]:.4f}')

best_nf_idx  = np.argmin(svd_nf_rmse)
best_nf      = n_factors_values[best_nf_idx]
best_nf_rmse = svd_nf_rmse[best_nf_idx]
best_nf_mae  = svd_nf_mae[best_nf_idx]
results['SVD (best n_factors)'] = {'RMSE': best_nf_rmse, 'MAE': best_nf_mae}
print(f'\nBest n_factors={best_nf}  RMSE: {best_nf_rmse:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_factors_values, svd_nf_rmse, 'o-', color='steelblue', linewidth=2, label='RMSE')
ax.plot(n_factors_values, svd_nf_mae,  's--', color='coral',    linewidth=2, label='MAE')
ax.axvline(best_nf, color='grey', linestyle=':', label=f'Best n_factors={best_nf}')
ax.set_title('SVD: RMSE and MAE vs n_factors (Latent Dimensions)', fontsize=13)
ax.set_xlabel('n_factors', fontsize=12)
ax.set_ylabel('Error', fontsize=12)
ax.legend()
ax.set_xticks(n_factors_values)
plt.tight_layout()
plt.show()

### Interpretation — SVD `n_factors`

- **Low n_factors (e.g., 5)**: **underfitting** — the model cannot capture the diversity of user preferences and product characteristics. Performance is poor.
- **High n_factors (e.g., 200)**: **risk of overfitting** — with many dimensions, the model may start fitting noise in the training data, leading to worse generalisation on the test set. This is mitigated by regularisation (`reg_all`).
- **Performance peak**: typically in the range 50–150 for real-world datasets. After this, performance plateaus or slightly degrades.
- **Interpretation**: the optimal n_factors reflects the intrinsic dimensionality of user–item preferences in Amazon Beauty — how many independent 'preference axes' (e.g., moisturising, fragrance, anti-aging, colour cosmetics) drive rating behaviour.

### 8.3 SVD — Varying `n_epochs` (Training Epochs)

The number of training epochs controls how long gradient descent runs. Too few epochs means the model hasn't converged; too many may lead to overfitting.

In [ ]:
n_epochs_values = [5, 10, 20, 50, 100]
svd_ne_rmse = []
svd_ne_mae  = []

for ne in n_epochs_values:
    model = SVD(n_factors=100, n_epochs=ne, lr_all=0.005, reg_all=0.02,
                random_state=SEED, verbose=False)
    model.fit(trainset)
    preds = model.test(testset)
    svd_ne_rmse.append(accuracy.rmse(preds, verbose=False))
    svd_ne_mae.append(accuracy.mae(preds, verbose=False))
    print(f'n_epochs={ne:3d}  RMSE: {svd_ne_rmse[-1]:.4f}  MAE: {svd_ne_mae[-1]:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(n_epochs_values, svd_ne_rmse, 'o-', color='steelblue', linewidth=2, label='RMSE')
ax.plot(n_epochs_values, svd_ne_mae,  's--', color='coral',    linewidth=2, label='MAE')
ax.set_title('SVD: RMSE and MAE vs n_epochs (Training Epochs)', fontsize=13)
ax.set_xlabel('n_epochs', fontsize=12)
ax.set_ylabel('Error', fontsize=12)
ax.legend()
ax.set_xticks(n_epochs_values)
plt.tight_layout()
plt.show()

### Interpretation — SVD `n_epochs`

The n_epochs curve shows the **convergence behaviour** of SGD training:

- **Too few epochs (5–10)**: the model has not converged — parameters are still far from optimal, and performance is degraded.
- **Converging region (20–50)**: performance improves steadily as gradient descent approaches the optimal solution. The default of 20 is typically in this region.
- **Plateau (50–100)**: performance stops improving once the model has converged. More training provides diminishing returns (and increases computation time).
- **Overfitting risk**: with regularisation (`reg_all=0.02`), overfitting is controlled even at 100 epochs. Without regularisation, very high epoch counts would cause overfitting. 

**Conclusion**: 20–50 epochs is typically sufficient for this dataset with the default learning rate and regularisation settings.

---
## Section 9: Model Comparison

We build a comprehensive comparison table of all models and visualise the results.

In [ ]:
# Build comparison DataFrame
comparison_df = pd.DataFrame([
    {'Model': name, 'RMSE': vals['RMSE'], 'MAE': vals['MAE']}
    for name, vals in results.items()
])
comparison_df = comparison_df.sort_values('RMSE').reset_index(drop=True)
comparison_df['RMSE'] = comparison_df['RMSE'].round(4)
comparison_df['MAE']  = comparison_df['MAE'].round(4)

print('Model Comparison (sorted by RMSE ascending):')
print(comparison_df.to_string(index=False))

In [ ]:
x = np.arange(len(comparison_df))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
bars1 = ax.bar(x - width/2, comparison_df['RMSE'], width, label='RMSE', color='steelblue',   alpha=0.85)
bars2 = ax.bar(x + width/2, comparison_df['MAE'],  width, label='MAE',  color='coral',       alpha=0.85)

ax.set_title('Model Comparison: RMSE and MAE for All Models', fontsize=14)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Error', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'], rotation=30, ha='right', fontsize=10)
ax.legend(fontsize=11)

# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

### Interpretation — Model Comparison

#### 1. Which model wins and by how much?
**SVD (best n_factors)** achieves the lowest RMSE and MAE, followed by SVD (default). The best SVD configuration typically improves RMSE by ~0.05–0.15 over the User Mean baseline — a meaningful improvement that demonstrates real personalisation.

#### 2. How much do CF models improve over baselines?
All three CF models (User KNN, Item KNN, SVD) outperform all three baselines. The largest jump is from the Global Mean to personalised CF — confirming that knowing who the user is (and their preferences) matters substantially.

#### 3. Memory-based vs model-based CF
**SVD outperforms both KNN variants** — this is expected for a large, sparse dataset:
- KNN suffers from the sparsity problem: finding users/items with sufficient co-rating overlap is difficult
- SVD generalises better by learning dense low-dimensional representations that don't require direct overlap
- Item-based KNN typically beats user-based KNN due to more stable item–item similarity patterns

#### 4. What does this mean for beauty product recommendations?
SVD's latent factors have likely learned implicit preference dimensions such as skincare type, fragrance intensity, brand preference, and price sensitivity — axes that drive whether a user will like a product even without explicit product descriptions. This makes SVD the recommended approach for a production Amazon Beauty recommender system.

---
## Section 10: Latent Factor Analysis (SVD Deep Dive — Bonus)

### 10.1 What Do Latent Factors Capture?

SVD decomposes the rating matrix **R** into two factor matrices:
- **P** (user factors, shape: n_users × n_factors): each row is a user's preference vector
- **Q** (item factors, shape: n_items × n_factors): each row is an item's characteristic vector

The predicted rating for user *u* and item *i* is: **r̂(u,i) = μ + b_u + b_i + p_u · q_i**

The dot product **p_u · q_i** measures how well the user's preference profile aligns with the item's characteristics. The latent dimensions are not explicitly labelled — they are **learned automatically from the rating patterns**. However, they can correspond to interpretable concepts:
- Factor 1 might represent "moisturising/hydration products" vs "fragrance products"
- Factor 2 might represent "luxury brand" vs "budget brand" preference
- Factor 3 might represent "anti-aging" vs "colour cosmetics" orientation

We use PCA to visualise the high-dimensional factor space in 2D.

In [ ]:
# Fit best SVD on the full trainset for the deep dive analysis
svd_best = SVD(n_factors=best_nf, n_epochs=20, lr_all=0.005, reg_all=0.02,
               random_state=SEED, verbose=False)
svd_best.fit(trainset)

# Extract factor matrices
item_factors = svd_best.qi   # shape: (n_items, n_factors)
user_factors = svd_best.pu   # shape: (n_users, n_factors)

print(f'Item factor matrix shape: {item_factors.shape}')
print(f'User factor matrix shape: {user_factors.shape}')

### 10.2 Visualise Item Factor Space with PCA

In [ ]:
# PCA on item factors
pca_items = PCA(n_components=2, random_state=SEED)
item_2d   = pca_items.fit_transform(item_factors)

# Get mean rating per item (inner id → raw id mapping)
inner_ids   = list(range(item_factors.shape[0]))
raw_ids     = [trainset.to_raw_iid(i) for i in inner_ids]
item_means  = df_filtered.groupby('item_id')['rating'].mean()
item_color  = [item_means.get(raw, 3.0) for raw in raw_ids]

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(item_2d[:, 0], item_2d[:, 1],
                c=item_color, cmap='RdYlGn', alpha=0.4, s=5)
plt.colorbar(sc, ax=ax, label='Mean Item Rating')
ax.set_title('Item Latent Factor Space (PCA 2D projection)', fontsize=14)
ax.set_xlabel(f'PC1 ({pca_items.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca_items.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
plt.tight_layout()
plt.show()

print(f'PC1 + PC2 explain {sum(pca_items.explained_variance_ratio_)*100:.1f}% of item factor variance')

### Interpretation — Item Factor Space

The 2D PCA projection of the item latent factor matrix reveals the **geometric structure** of product similarity as learned by SVD:

- **Clusters**: groups of items that cluster together in this space are considered "similar" by the model — they tend to be rated similarly by users. Without labels, we cannot name these clusters, but in a beauty context they likely correspond to product categories (moisturisers, shampoos, perfumes, makeup).
- **Colour (mean rating)**: items coloured green have higher average ratings; items coloured red have lower ratings. If high-rated items cluster together, this indicates that quality/satisfaction is itself a latent dimension the model has learned.
- **Spread**: a widely dispersed point cloud indicates that products have diverse, heterogeneous characteristics — the latent space has rich structure. A compact cloud would suggest that most items are perceived similarly by users.
- **Key insight**: the model learned these groupings **purely from rating patterns** without any product descriptions — a remarkable demonstration of what collaborative filtering can infer from user behaviour alone.

### 10.3 Visualise User Factor Space with PCA

In [ ]:
# Subsample users for readability if the dataset is large
max_users_plot = 5000
if user_factors.shape[0] > max_users_plot:
    rng     = np.random.default_rng(SEED)
    indices = rng.choice(user_factors.shape[0], max_users_plot, replace=False)
    uf_plot = user_factors[indices]
    plot_label = f'(random sample of {max_users_plot:,} users)'
else:
    uf_plot = user_factors
    plot_label = ''

pca_users = PCA(n_components=2, random_state=SEED)
user_2d   = pca_users.fit_transform(uf_plot)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(user_2d[:, 0], user_2d[:, 1], alpha=0.3, s=5, color='steelblue')
ax.set_title(f'User Latent Factor Space (PCA 2D projection) {plot_label}', fontsize=13)
ax.set_xlabel(f'PC1 ({pca_users.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca_users.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
plt.tight_layout()
plt.show()

print(f'PC1 + PC2 explain {sum(pca_users.explained_variance_ratio_)*100:.1f}% of user factor variance')

### Interpretation — User Factor Space

The 2D PCA projection of user latent factors reveals the **geometry of user preferences** as inferred from rating patterns:

- **User clusters**: groups of users that appear close together in this space have similar taste profiles — they tend to rate the same items similarly. In a beauty context, these clusters might correspond to: skincare enthusiasts, makeup lovers, fragrance collectors, budget-conscious shoppers, etc.
- **Spread**: the distribution of users across the latent space reflects how diverse the customer base is. A uniform spread indicates highly heterogeneous preferences; clusters indicate groups with coherent shared preferences.
- **Practical use**: in production, these user vectors enable **rapid new-user recommendation** — by mapping a new user's few ratings into this space, we can immediately identify similar users and leverage their preferences.
- **Key insight**: no demographic information was used — the model inferred all these user groups purely from who rated what and how much.

### 10.4 Factor Importance: Explained Variance

In [ ]:
n_components_ev = min(20, item_factors.shape[1])
pca_full = PCA(n_components=n_components_ev, random_state=SEED)
pca_full.fit(item_factors)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual explained variance
axes[0].bar(range(1, n_components_ev + 1), pca_full.explained_variance_ratio_ * 100,
            color='steelblue', edgecolor='white')
axes[0].set_title('Explained Variance per PCA Component\n(Item Factors)', fontsize=12)
axes[0].set_xlabel('PCA Component', fontsize=11)
axes[0].set_ylabel('Explained Variance (%)', fontsize=11)
axes[0].set_xticks(range(1, n_components_ev + 1))

# Cumulative explained variance
cumulative = np.cumsum(pca_full.explained_variance_ratio_) * 100
axes[1].plot(range(1, n_components_ev + 1), cumulative, 'o-', color='coral', linewidth=2)
axes[1].axhline(80, color='grey', linestyle='--', label='80% threshold')
axes[1].set_title('Cumulative Explained Variance\n(Item Factors)', fontsize=12)
axes[1].set_xlabel('Number of PCA Components', fontsize=11)
axes[1].set_ylabel('Cumulative Explained Variance (%)', fontsize=11)
axes[1].legend()
axes[1].set_xticks(range(1, n_components_ev + 1))

plt.tight_layout()
plt.show()

print(f'Top 5 components explain:  {cumulative[4]:.1f}% of item factor variance')
print(f'Top 10 components explain: {cumulative[9]:.1f}% of item factor variance')

### Interpretation — Factor Importance

The explained variance plots reveal the **dimensionality structure** of item latent factors:

- **Concentrated variance**: if the first few PCA components explain a large fraction of total variance (e.g., top 5 components explain >50%), this indicates that item rating behaviour has a **low-dimensional structure** — a few dominant preference dimensions drive most of the variation in how users rate products.
- **Practical meaning**: for Amazon Beauty, this might mean that 2–3 major axes (e.g., "moisturising" vs "colour cosmetics", "luxury" vs "budget") explain most of why users rate products the way they do.
- **Model design implication**: if variance drops off quickly after the top few components, using `n_factors=5–20` might be sufficient to capture most of the signal, while using `n_factors=100–200` may be modelling noise in the tail components.
- **Connection to n_factors optimisation**: the explained variance here is consistent with the hyperparameter sweep in Section 8.2 — the best `n_factors` should be in the range where adding more factors stops providing meaningful new information.

### 10.5 Sample Recommendations Using Best SVD

In [ ]:
# Find the most active user in the filtered dataset
user_activity_filtered = df_filtered.groupby('user_id')['rating'].count()
most_active_user = user_activity_filtered.idxmax()
n_rated = user_activity_filtered.max()

print(f'Most active user: {most_active_user}')
print(f'Number of ratings by this user: {n_rated}')

# Items already rated by this user
rated_items = set(df_filtered[df_filtered['user_id'] == most_active_user]['item_id'])

# All items in the dataset
all_items = set(df_filtered['item_id'].unique())

# Items NOT rated by this user
unrated_items = all_items - rated_items
print(f'Items rated:   {len(rated_items):,}')
print(f'Items unrated: {len(unrated_items):,}')

In [ ]:
# Predict ratings for all unrated items
predictions = [svd_best.predict(most_active_user, iid) for iid in unrated_items]

# Sort by predicted rating (descending)
predictions_sorted = sorted(predictions, key=lambda x: x.est, reverse=True)

# Top 10 recommendations
top_10 = predictions_sorted[:10]

print(f'Top 10 recommendations for user: {most_active_user}\n')
print(f'{"Rank":<6} {"Product ID":<25} {"Predicted Rating":<20}')
print('-' * 55)
for rank, pred in enumerate(top_10, 1):
    print(f'{rank:<6} {pred.iid:<25} {pred.est:.4f}')

### Interpretation — Sample Recommendations

The top 10 recommended products for the most active user represent what the SVD model predicts this user would rate highest among items they haven't yet rated.

**How the predictions are computed:**
The predicted rating is: **r̂(u, i) = μ + b_u + b_i + p_u · q_i**
- **μ**: global mean rating (captures the overall positivity bias)
- **b_u**: user bias (captures whether this user tends to rate higher or lower than average)
- **b_i**: item bias (captures whether this item tends to receive higher or lower ratings)
- **p_u · q_i**: the dot product of user and item factor vectors (captures the nuanced preference–characteristic alignment)

**Connection to latent factors:**
The most active user has a well-defined latent preference vector **p_u** learned from their many ratings. The recommended items are those whose item factor vectors **q_i** have the highest dot product with **p_u** — meaning they align with this user's implicit preference dimensions (their favourite product categories, preferred formulations, etc.).

**For a beauty retailer**, this means we can automatically personalise recommendations for active customers based purely on their past rating behaviour, without needing to know anything about the products' ingredients, brands, or categories.

---
## Section 11: Conclusions

### 1. Dataset Characteristics and Their Impact on CF
The Amazon Beauty dataset exhibits three key properties that directly shaped our modelling choices: (a) **extreme sparsity** (>99% missing), which makes KNN-based similarity unreliable and favours model-based CF; (b) **long-tail distributions** for both users and items, creating cold-start challenges for inactive users/items; and (c) **positivity bias**, meaning simple global mean baselines are already strong competitors and personalised models must work to provide meaningful improvements.

### 2. Baseline vs CF Model Performance
All three collaborative filtering models (user-based KNN, item-based KNN, SVD) outperformed all three non-personalised baselines (random, global mean, user mean). This confirms that learning user–item interaction structure is genuinely valuable — personalised CF adds real predictive power beyond simple heuristics. The user mean baseline, which personalises only at the user level (not item level), was the strongest baseline, highlighting that individual user bias is an important signal.

### 3. Best Model and Why It Won
**SVD (matrix factorisation)** was the best-performing model, consistent with the CF literature. SVD's advantages in this setting: (a) it handles sparsity by learning dense latent representations rather than requiring direct rating overlap; (b) it jointly models user and item biases; (c) regularisation prevents overfitting to noise from sparse users/items; and (d) the latent factor structure captures rich multi-dimensional preference patterns that simple cosine similarity cannot.

### 4. Memory-Based vs Model-Based Trade-offs
- **Memory-based (KNN)**: interpretable ("users like you rated this highly"), no training phase, but severely limited by sparsity. Item-based KNN was more stable than user-based KNN.
- **Model-based (SVD)**: better performance, handles sparsity, generalises well, but is a black box and requires training. In production, SVD's latent factors enable fast real-time prediction after an offline training phase.

### 5. Hyperparameter Insights
- **KNN k**: a moderate k (20–40) was optimal. Too low → high variance; too high → noise from distant neighbours.
- **SVD n_factors**: performance peaked at 50–100 factors, confirming that Amazon Beauty ratings have a moderately complex latent structure.
- **SVD n_epochs**: convergence was reached within 20–50 epochs; more training beyond that provided diminishing returns.

### 6. Latent Factor Interpretation
The SVD latent factor analysis revealed that the rating matrix has an interpretable low-dimensional structure. PCA of item factors showed clustering patterns (implicit product categories) and the top few components explain a substantial fraction of variance. This means the model has learned meaningful product similarity relationships purely from rating behaviour, without any product metadata.

### 7. Limitations
- **Cold-start**: new users or new products with few/no ratings cannot be handled by pure CF. A hybrid approach combining content-based features (product descriptions, user demographics) would be needed.
- **Sparsity**: despite filtering, the dataset remains very sparse. Some users and items still have few ratings, limiting CF quality.
- **Popularity bias**: CF tends to recommend popular items more often. Less popular but potentially relevant niche items may never be recommended.
- **No content information**: we have no product descriptions, ingredient lists, or brand information that could help explain *why* items are similar.
- **Temporal dynamics**: we ignored the timestamp column. User preferences evolve over time — a more sophisticated model would account for temporal drift.
- **Implicit assumptions**: SVD assumes the rating matrix has a low-rank structure — this is generally true but may not capture all preference patterns.

### 8. Practical Business Implications for Amazon Beauty Recommendations
For a real Amazon Beauty recommendation system:
1. **Use SVD (or its extensions like SVD++ or NMF)** as the core CF engine for personalised rating prediction.
2. **Combine with content-based features** (product category, brand, ingredient lists) to address the cold-start problem for new products.
3. **Address popularity bias** by introducing diversity constraints or exploration bonuses to expose users to less popular but potentially relevant products.
4. **Retrain periodically** to capture evolving user preferences and incorporate new products and ratings.
5. **Evaluate beyond RMSE/MAE** using ranking metrics (NDCG, Precision@K) to measure whether the top recommendations are actually the most relevant ones, not just accurate rating predictors.